In [ ]:
import os
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121, InceptionV3, MobileNetV2, ResNet50
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, log_loss

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path(".").resolve()
SPLIT_DIR = PROJECT_DIR / "data_split"
MODEL_WEIGHTS_DIR = PROJECT_DIR / "model_weights"
BENCHMARK_CSV = PROJECT_DIR / "benchmark.csv"

CLASS_NAMES = ["Normal", "Stroke"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 8

print("TensorFlow:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
print("Single benchmark file:", BENCHMARK_CSV)


In [ ]:
def make_data_augmentation():
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.03),
            layers.RandomZoom(0.08),
            layers.RandomContrast(0.08),
        ],
        name="augmentation",
    )


def load_split_dataset(split_name: str, shuffle: bool):
    ds = tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split_name,
        labels="inferred",
        label_mode="binary",
        class_names=CLASS_NAMES,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
        shuffle=shuffle,
    )
    if split_name != "train":
        ds = ds.cache()
    return ds.prefetch(AUTOTUNE)


train_ds = load_split_dataset("train", shuffle=True)
val_ds = load_split_dataset("val", shuffle=False)
test_ds = load_split_dataset("test", shuffle=False)

print("Class names:", CLASS_NAMES)


In [ ]:
TRANSFER_SPECS = {
    "ResNet50": (ResNet50, resnet_preprocess),
    "DenseNet121": (DenseNet121, densenet_preprocess),
    "MobileNetV2": (MobileNetV2, mobilenet_preprocess),
    "InceptionV3": (InceptionV3, inception_preprocess),
}


def build_transfer_model(model_name: str):
    base_constructor, preprocess_fn = TRANSFER_SPECS[model_name]
    base_model = base_constructor(
        include_top=False,
        weights="imagenet",
        input_shape=IMAGE_SIZE + (3,),
    )
    base_model.trainable = False

    inputs = keras.Input(shape=IMAGE_SIZE + (3,), name=f"{model_name}_input")
    x = make_data_augmentation()(inputs)
    x = layers.Lambda(preprocess_fn, name=f"{model_name}_preprocess")(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name=f"{model_name}_gap")(x)
    x = layers.Dropout(0.3, name=f"{model_name}_dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32", name=f"{model_name}_output")(x)
    return keras.Model(inputs, outputs, name=model_name)


def build_alexnet():
    return keras.Sequential(
        [
            layers.Input(shape=IMAGE_SIZE + (3,)),
            layers.Rescaling(1.0 / 255.0),
            make_data_augmentation(),
            layers.Conv2D(96, 11, strides=4, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(3, strides=2),
            layers.Conv2D(256, 5, padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(3, strides=2),
            layers.Conv2D(384, 3, padding="same", activation="relu"),
            layers.Conv2D(384, 3, padding="same", activation="relu"),
            layers.Conv2D(256, 3, padding="same", activation="relu"),
            layers.MaxPooling2D(3, strides=2),
            layers.Flatten(),
            layers.Dense(2048, activation="relu"),
            layers.Dropout(0.5),
            layers.Dense(512, activation="relu"),
            layers.Dropout(0.4),
            layers.Dense(1, activation="sigmoid", dtype="float32"),
        ],
        name="AlexNet",
    )


def build_model(model_name: str):
    if model_name == "AlexNet":
        return build_alexnet()
    return build_transfer_model(model_name)


def compile_model(model: keras.Model, learning_rate: float = 1e-4):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )


BENCHMARK_MODEL_NAMES = ["AlexNet", "ResNet50", "DenseNet121", "MobileNetV2", "InceptionV3"]
print("Benchmark base models:", BENCHMARK_MODEL_NAMES)


In [ ]:
def collect_labels(dataset):
    labels = []
    for _, y in dataset.as_numpy_iterator():
        labels.append(np.asarray(y).reshape(-1))
    return np.concatenate(labels).astype(np.int32)


def predict_probabilities(model, dataset):
    return model.predict(dataset, verbose=0).reshape(-1).astype(np.float32)


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(np.int32)
    y_prob = np.clip(np.asarray(y_prob).astype(np.float32), 1e-7, 1 - 1e-7)
    y_pred = (y_prob >= threshold).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "Sensitivity": sensitivity,
        "AUC": roc_auc_score(y_true, y_prob),
        "Loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def build_benchmark_model(model_name: str):
    if model_name == "HybridFusion_R50_AlexNet":
        if "build_hybrid_fusion_r50_alexnet" not in globals():
            raise RuntimeError("Hybrid builder is not available yet. Run Cell 6 before benchmarking.")
        return build_hybrid_fusion_r50_alexnet()
    return build_model(model_name)


def evaluate_model_from_saved_weights(model_name: str):
    weights_path = MODEL_WEIGHTS_DIR / f"{model_name}.weights.h5"
    if not weights_path.exists():
        raise FileNotFoundError(str(weights_path))

    model = build_benchmark_model(model_name)
    compile_model(model, learning_rate=1e-4)
    model.load_weights(str(weights_path))

    y_true = collect_labels(test_ds)
    y_prob = predict_probabilities(model, test_ds)
    metrics = compute_binary_metrics(y_true, y_prob, threshold=0.5)

    row = {
        "Model": model_name,
        **metrics,
    }

    keras.backend.clear_session()
    gc.collect()
    return row


def run_full_benchmark_from_weights(model_names=None):
    names = list(model_names) if model_names is not None else [
        "AlexNet",
        "ResNet50",
        "DenseNet121",
        "MobileNetV2",
        "InceptionV3",
        "HybridFusion_R50_AlexNet",
    ]

    rows = []
    missing = []
    failed = []

    for model_name in names:
        print("=" * 80)
        print(f"Evaluating from weights: {model_name}")
        try:
            rows.append(evaluate_model_from_saved_weights(model_name))
        except FileNotFoundError as err:
            missing.append((model_name, str(err)))
            print(f"missing weights, skipping: {err}")
        except Exception as err:
            failed.append((model_name, str(err)))
            print(f"failed, skipping: {err}")

    if not rows:
        raise RuntimeError("No models were evaluated. Check model_weights files and notebook order.")

    benchmark_df = pd.DataFrame(rows).sort_values(["AUC", "Accuracy", "F1"], ascending=False).reset_index(drop=True)
    benchmark_df.to_csv(BENCHMARK_CSV, index=False)

    print("Saved benchmark file:", BENCHMARK_CSV)
    if missing:
        print("Missing weights:")
        for model_name, path in missing:
            print(f"- {model_name}: {path}")
    if failed:
        print("Failed models:")
        for model_name, msg in failed:
            print(f"- {model_name}: {msg}")

    return benchmark_df


print("Benchmark utility ready. Run Cell 6, then call run_full_benchmark_from_weights().")

In [ ]:
def get_layer_recursive(model, layer_name):
    for layer in model._flatten_layers(include_self=False, recursive=True):
        if layer.name == layer_name:
            return layer
    raise ValueError(f"Layer {layer_name} not found")


def list_conv_layer_names(model):
    return [
        layer.name
        for layer in model._flatten_layers(include_self=False, recursive=True)
        if isinstance(layer, layers.Conv2D)
    ]


def choose_gradcam_layer(model, model_name):
    conv_names = list_conv_layer_names(model)
    if model_name == "AlexNet" and len(conv_names) >= 2:
        return conv_names[-2]
    if conv_names:
        return conv_names[-1]
    return None


def load_image_for_model(image_path):
    image = keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image_array = keras.utils.img_to_array(image)
    image_tensor = np.expand_dims(image_array, axis=0)
    return image_array.astype("uint8"), image_tensor


def make_gradcampp_heatmap(image_tensor, model, layer_name):
    target_layer = get_layer_recursive(model, layer_name)
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[target_layer.output, model.outputs[0]]
    )
    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                conv_out, preds = grad_model(image_tensor, training=False)
                tape0.watch(conv_out)
                tape1.watch(conv_out)
                tape2.watch(conv_out)
                score = preds[:, 0]
            first = tape0.gradient(score, conv_out)
        second = tape1.gradient(first, conv_out)
    third = tape2.gradient(second, conv_out)
    global_sum = tf.reduce_sum(conv_out, axis=(1, 2), keepdims=True)
    alpha_num = second[0]
    alpha_den = 2.0 * second[0] + third[0] * global_sum[0] + 1e-7
    alpha = alpha_num / alpha_den
    alpha = tf.nn.relu(alpha)
    weights = tf.reduce_sum(alpha * tf.nn.relu(first[0]), axis=(0, 1))
    heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    hmax = tf.reduce_max(heatmap)
    heatmap = tf.where(hmax > 0, heatmap / hmax, heatmap)
    return heatmap.numpy()


def overlay_heatmap(image_array, heatmap, alpha=0.62):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (image_array.shape[0], image_array.shape[1])).numpy().squeeze()
    heatmap_resized = np.clip(heatmap_resized, 0.0, 1.0)
    colored_heatmap = cm.jet(heatmap_resized)[..., :3].astype("float32")
    image_float = image_array.astype("float32") / 255.0
    blended = np.clip((1 - alpha) * image_float + alpha * colored_heatmap, 0.0, 1.0)
    return (255.0 * blended).astype("uint8"), heatmap_resized


def connected_components(mask):
    h, w = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    components = []
    for y in range(h):
        for x in range(w):
            if not mask[y, x] or visited[y, x]:
                continue
            stack = [(y, x)]
            visited[y, x] = True
            pixels = []
            while stack:
                cy, cx = stack.pop()
                pixels.append((cy, cx))
                for ny in range(max(0, cy - 1), min(h, cy + 2)):
                    for nx in range(max(0, cx - 1), min(w, cx + 2)):
                        if not visited[ny, nx] and mask[ny, nx]:
                            visited[ny, nx] = True
                            stack.append((ny, nx))
            components.append(pixels)
    return components


def extract_roi_bboxes(heatmap_resized, threshold=0.30, percentile=84, max_regions=3):
    max_value = float(np.max(heatmap_resized))
    if max_value <= 1e-8:
        return []
    positive = heatmap_resized[heatmap_resized > 0]
    if positive.size < 20:
        return []
    cutoff = max(threshold * max_value, np.percentile(positive, percentile))
    comps = connected_components(heatmap_resized >= cutoff)
    candidates = []
    for pixels in comps:
        ys = np.array([p[0] for p in pixels])
        xs = np.array([p[1] for p in pixels])
        if len(xs) < 4:
            continue
        score = float(np.sum(heatmap_resized[ys, xs]))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
        candidates.append((score, bbox))
    candidates.sort(key=lambda x: x[0], reverse=True)
    return [bbox for _, bbox in candidates[:max_regions]]

In [ ]:
#scipy tissue-masking imports (not in cell 1)
from scipy.ndimage import binary_fill_holes, binary_opening, binary_closing, gaussian_filter

#---tissue masking helpers---

def largest_component(mask):
    comps = connected_components(mask)
    if not comps:
        return mask
    largest = max(comps, key=len)
    out = np.zeros_like(mask, dtype=bool)
    ys = np.array([p[0] for p in largest])
    xs = np.array([p[1] for p in largest])
    out[ys, xs] = True
    return out


def make_intracranial_mask(image_array):
    #threshold on grayscale mean, keep largest filled region
    gray = image_array.astype(np.float32).mean(axis=-1)
    nonzero = gray[gray > 0]
    if nonzero.size < 20:
        return np.ones_like(gray, dtype=bool)
    cutoff = np.percentile(nonzero, 25)
    raw = gray > cutoff
    raw = binary_opening(raw, structure=np.ones((3, 3), dtype=bool))
    raw = largest_component(raw)
    raw = binary_fill_holes(raw)
    raw = binary_closing(raw, structure=np.ones((5, 5), dtype=bool))
    return raw.astype(bool)


def make_brain_tissue_mask(image_array, intracranial_mask):
    #exclude skull-bright and near-zero voxels inside the cranial region
    gray = image_array.astype(np.float32).mean(axis=-1)
    inside_vals = gray[intracranial_mask]
    if inside_vals.size < 20:
        return intracranial_mask
    low = np.percentile(inside_vals, 12)
    high = np.percentile(inside_vals, 96)
    tissue = intracranial_mask & (gray >= low) & (gray <= high)
    tissue = binary_opening(tissue, structure=np.ones((3, 3), dtype=bool))
    tissue = binary_closing(tissue, structure=np.ones((5, 5), dtype=bool))
    tissue = largest_component(tissue)
    return tissue.astype(bool)


def smooth_heatmap(heatmap_resized, sigma=1.4):
    #gaussian smoothing to reduce noise before thresholding
    return gaussian_filter(np.asarray(heatmap_resized, dtype=np.float32), sigma=sigma)


def outside_heat_ratio(heatmap_resized, intracranial_mask):
    #fraction of total activation energy that falls outside the brain mask
    heat = np.asarray(heatmap_resized, dtype=np.float32)
    total = float(np.sum(heat)) + 1e-8
    outside = float(np.sum(heat * (~intracranial_mask).astype(np.float32)))
    return outside / total


def activation_concentration(masked_heatmap):
    #gini-like score: higher means activation is more focal (less spread)
    flat = np.sort(masked_heatmap.ravel().astype(np.float32))
    total = float(flat.sum()) + 1e-8
    n = flat.size
    cumsum = np.cumsum(flat)
    #lorenz area above diagonal -> concentration
    gini = 1.0 - 2.0 * float(cumsum.sum()) / (total * n)
    return float(np.clip(gini, 0.0, 1.0))


def extract_roi_bboxes_in_tissue(heatmap_resized, tissue_mask, threshold=0.35, percentile=88, max_regions=4):
    #returns bounding boxes of high-activation connected components strictly inside tissue
    heat = np.asarray(heatmap_resized, dtype=np.float32)
    mask = np.asarray(tissue_mask, dtype=bool)
    masked_heat = heat * mask.astype(np.float32)

    max_value = float(np.max(masked_heat))
    if max_value <= 1e-8:
        return []
    positive = masked_heat[masked_heat > 0]
    if positive.size < 20:
        return []

    cutoff = max(threshold * max_value, np.percentile(positive, percentile))
    candidate_mask = masked_heat >= cutoff
    comps = connected_components(candidate_mask)

    h, w = masked_heat.shape
    min_area = max(20, int(0.0015 * h * w))
    border_margin = max(6, int(0.04 * min(h, w)))
    candidates = []

    for pixels in comps:
        if len(pixels) < min_area:
            continue
        ys = np.array([p[0] for p in pixels])
        xs = np.array([p[1] for p in pixels])
        if float(np.mean(mask[ys, xs])) < 0.98:
            continue
        if (xs.min() <= border_margin or ys.min() <= border_margin
                or xs.max() >= (w - border_margin) or ys.max() >= (h - border_margin)):
            continue
        score = float(np.sum(masked_heat[ys, xs]))
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
        candidates.append((score, bbox))

    candidates.sort(key=lambda x: x[0], reverse=True)
    return [bbox for _, bbox in candidates[:max_regions]]


#---hybrid model definition---

def build_hybrid_fusion_r50_alexnet():          #hybrdid novelty
    resnet_backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=IMAGE_SIZE + (3,),
    )
    resnet_backbone.trainable = False

    inputs = keras.Input(shape=IMAGE_SIZE + (3,), name="HybridFusion_input")
    augmented = make_data_augmentation()(inputs)

    a = layers.Lambda(resnet_preprocess, name="hybrid_resnet_preprocess")(augmented)
    a = resnet_backbone(a, training=False)
    a = layers.GlobalAveragePooling2D(name="hybrid_resnet_gap")(a)

    b = layers.Rescaling(1.0 / 255.0, name="hybrid_alex_rescale")(augmented)
    b = layers.Conv2D(96, 11, strides=4, padding="same", activation="relu", name="hybrid_alex_conv1")(b)
    b = layers.BatchNormalization(name="hybrid_alex_bn1")(b)
    b = layers.MaxPooling2D(3, strides=2, name="hybrid_alex_pool1")(b)
    b = layers.Conv2D(256, 5, padding="same", activation="relu", name="hybrid_alex_conv2")(b)
    b = layers.BatchNormalization(name="hybrid_alex_bn2")(b)
    b = layers.MaxPooling2D(3, strides=2, name="hybrid_alex_pool2")(b)
    b = layers.Conv2D(256, 3, padding="same", activation="relu", name="hybrid_alex_conv3")(b)
    b = layers.GlobalAveragePooling2D(name="hybrid_alex_gap")(b)

    merged = layers.Concatenate(name="hybrid_concat")([a, b])
    gate = layers.Dense(merged.shape[-1], activation="sigmoid", name="hybrid_gate")(merged)
    fused = layers.Multiply(name="hybrid_gated_fusion")([merged, gate])

    x = layers.Dense(512, activation="relu", name="hybrid_fc1")(fused)
    x = layers.Dropout(0.35, name="hybrid_dropout1")(x)
    x = layers.Dense(128, activation="relu", name="hybrid_fc2")(x)
    x = layers.Dropout(0.25, name="hybrid_dropout2")(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32", name="hybrid_output")(x)

    return keras.Model(inputs, outputs, name="HybridFusion_R50_AlexNet")


#---load hybrid model---

HYBRID_NAME = "HybridFusion_R50_AlexNet"
hybrid_weights = MODEL_WEIGHTS_DIR / f"{HYBRID_NAME}.weights.h5"

hybrid_model = build_hybrid_fusion_r50_alexnet()
compile_model(hybrid_model, learning_rate=1e-4)

if hybrid_weights.exists():
    hybrid_model.load_weights(str(hybrid_weights))
    print(f"loaded hybrid weights: {hybrid_weights}")
else:
    print("hybrid weights missing — retraining...")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=1, min_lr=1e-6, verbose=1),
    ]
    hybrid_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)
    hybrid_model.save_weights(str(hybrid_weights))
    print(f"saved retrained hybrid weights: {hybrid_weights}")

#grad-cam target layer for hybrid
HYBRID_LAYER = choose_gradcam_layer(hybrid_model, HYBRID_NAME)
print(f"grad-cam layer: {HYBRID_LAYER}")

In [ ]:
# Final presentation: exactly 1 normal + 1 stroke (ROI derived from focused hotspot)
from matplotlib.patches import Rectangle

run_seed = int.from_bytes(os.urandom(8), "big")
rng = random.Random(run_seed)

if "_previous_cases" not in globals():
    _previous_cases = {"normal": None, "stroke": None}

valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def get_images(class_name):
    return sorted([
        p for p in (SPLIT_DIR / "test" / class_name).iterdir()
        if p.suffix.lower() in valid_ext
    ])


def focused_hotspot(masked_heat, percentile=88):
    if np.any(masked_heat > 0):
        active = masked_heat[masked_heat > 0]
        cutoff = np.percentile(active, percentile)
        return np.where(masked_heat >= cutoff, masked_heat, 0.0).astype(np.float32)
    return masked_heat.astype(np.float32)


def roi_from_hotspot(hotspot, tissue_mask):
    rois = extract_roi_bboxes_in_tissue(
        hotspot,
        tissue_mask,
        threshold=0.45,
        percentile=92,
        max_regions=1,
    )
    return rois


def roi_heat_agreement(rois, heat):
    if not rois:
        return 0.0
    total = float(np.sum(heat)) + 1e-8
    inside = 0.0
    for (x1, y1, x2, y2) in rois:
        inside += float(np.sum(heat[y1:y2 + 1, x1:x2 + 1]))
    return inside / total


stroke_paths = get_images("Stroke")
normal_paths = get_images("Normal")

if not stroke_paths:
    raise ValueError("no stroke images found")
if not normal_paths:
    raise ValueError("no normal images found")

# Normal: choose one with lowest p(stroke).
normal_pool = rng.sample(normal_paths, k=min(30, len(normal_paths)))
normal_scored = []
for image_path in normal_pool:
    image_array, image_tensor = load_image_for_model(image_path)
    prob = float(hybrid_model.predict(image_tensor, verbose=0).ravel()[0])
    normal_scored.append({"path": image_path, "prob": prob, "image_array": image_array})
normal_scored.sort(key=lambda x: x["prob"])

prev_normal_name = _previous_cases.get("normal")
normal_case = next((c for c in normal_scored if c["path"].name != prev_normal_name), normal_scored[0])

# Stroke: choose one where displayed hotspot and ROI are aligned.
stroke_pool = rng.sample(stroke_paths, k=min(120, len(stroke_paths)))
stroke_scored = []
for image_path in stroke_pool:
    image_array, image_tensor = load_image_for_model(image_path)
    prob = float(hybrid_model.predict(image_tensor, verbose=0).ravel()[0])
    if prob < 0.60:
        continue

    raw_heatmap = make_gradcampp_heatmap(image_tensor, hybrid_model, HYBRID_LAYER)
    _, heatmap_resized = overlay_heatmap(image_array, raw_heatmap, alpha=0.62)
    heatmap_smooth = smooth_heatmap(heatmap_resized, sigma=1.2)

    intracranial_mask = make_intracranial_mask(image_array)
    tissue_mask = make_brain_tissue_mask(image_array, intracranial_mask)
    masked_heat = heatmap_smooth * tissue_mask.astype(np.float32)

    outside_ratio = outside_heat_ratio(heatmap_smooth, intracranial_mask)
    conc = activation_concentration(masked_heat)

    hotspot = focused_hotspot(masked_heat, percentile=88)
    rois = roi_from_hotspot(hotspot, tissue_mask)
    roi_heat = roi_heat_agreement(rois, hotspot)

    quality = prob * (1.0 - outside_ratio) * (0.35 + 0.65 * conc) * (0.35 + 0.65 * roi_heat)

    stroke_scored.append({
        "path": image_path,
        "prob": prob,
        "outside_ratio": outside_ratio,
        "conc": conc,
        "roi_heat": roi_heat,
        "quality": quality,
        "rois": rois,
        "image_array": image_array,
        "hotspot": hotspot,
    })

if not stroke_scored:
    raise RuntimeError("no stroke candidates passed probability gate")

stroke_scored.sort(key=lambda x: -x["quality"])

strict = [
    c for c in stroke_scored
    if c["prob"] >= 0.80
    and c["outside_ratio"] <= 0.42
    and c["roi_heat"] >= 0.20
    and len(c["rois"]) == 1
]

relaxed = [
    c for c in stroke_scored
    if c["prob"] >= 0.72
    and c["outside_ratio"] <= 0.55
    and c["roi_heat"] >= 0.14
    and len(c["rois"]) == 1
]

if strict:
    candidates = strict
elif relaxed:
    candidates = relaxed
else:
    with_roi = [c for c in stroke_scored if len(c["rois"]) == 1]
    candidates = with_roi if with_roi else stroke_scored

prev_stroke_name = _previous_cases.get("stroke")
stroke_case = next((c for c in candidates if c["path"].name != prev_stroke_name), candidates[0])

_previous_cases["normal"] = normal_case["path"].name
_previous_cases["stroke"] = stroke_case["path"].name

hotspot = stroke_case["hotspot"]
hmax = float(hotspot.max())
hotspot_norm = hotspot / hmax if hmax > 0 else hotspot
colored = plt.cm.hot(hotspot_norm)[..., :3].astype("float32")
img_float = stroke_case["image_array"].astype("float32") / 255.0
alpha_map = (hotspot_norm > 0.05).astype("float32")[..., np.newaxis] * 0.74
overlay = np.clip(img_float * (1.0 - alpha_map) + colored * alpha_map, 0.0, 1.0)
overlay = (overlay * 255).astype("uint8")

# Output 1: normal
fig_n, ax_n = plt.subplots(1, 1, figsize=(5, 5), dpi=115)
ax_n.imshow(normal_case["image_array"], cmap="gray")
ax_n.axis("off")
ax_n.set_title(
    f"NORMAL detected | p(stroke)={normal_case['prob']:.4f}\nNo lesion ROI highlighted",
    fontsize=11,
    color="green",
)
fig_n.tight_layout()
plt.show()

# Output 2: stroke with ROI + focused heatmap + overlay
fig_s, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=115)

axes[0].imshow(stroke_case["image_array"], cmap="gray")
axes[0].axis("off")
axes[0].set_title("Stroke: Original + ROI", fontsize=10)

for i, (x1, y1, x2, y2) in enumerate(stroke_case["rois"], start=1):
    axes[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="cyan", linewidth=2))
    axes[0].text(x1, max(8, y1 - 5), f"R{i}", color="cyan", fontsize=8, weight="bold")

im = axes[1].imshow(hotspot, cmap="hot", vmin=0, vmax=hmax if hmax > 0 else 1)
axes[1].axis("off")
axes[1].set_title("Stroke: Focused heatmap", fontsize=10)
for (x1, y1, x2, y2) in stroke_case["rois"]:
    axes[1].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="cyan", linewidth=1.5))
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(overlay)
axes[2].axis("off")
axes[2].set_title(
    f"STROKE detected | p(stroke)={stroke_case['prob']:.4f}",
    fontsize=10,
    color="darkred",
)
for (x1, y1, x2, y2) in stroke_case["rois"]:
    axes[2].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="cyan", linewidth=1.5))

fig_s.suptitle(
    f"{stroke_case['path'].name} | qualitative attention and ROI (not pixel-level lesion segmentation)",
    fontsize=8,
    color="gray",
)
fig_s.tight_layout()
plt.show()

print(f"run_seed={run_seed}")
print("Normal case:")
print(f"file={normal_case['path'].name} | predicted=NORMAL | p(stroke)={normal_case['prob']:.4f}")
print("Stroke case:")
print(
    f"file={stroke_case['path'].name} | predicted=STROKE | p(stroke)={stroke_case['prob']:.4f} | "
    f"rois={len(stroke_case['rois'])} | outside_ratio={stroke_case['outside_ratio']:.3f} | "
    f"roi_heat={stroke_case['roi_heat']:.3f}"
)